# Демо 1. Класичне донавчання: зміна голови в моделі BERT

У цьому ноутбуці донавчаємо модель **BERT** для задачі **визначення парафраз**: модель отримує два речення й визначає, чи передають вони той самий зміст.

План:
1. Завантажити датасет MRPC і токенізувати пари речень.
2. Завантажити BERT із **новою головою**-класифікатором.
3. **Повне донавчання** з `Trainer`: тренуються всі ваги (кістяк і голова).
4. Перевірити модель на власних прикладах.

Приклад за мотивами розділу 3 курсу [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter3/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

In [2]:
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

In [3]:
# Фіксуємо seed, щоб результати повторювалися між запусками
set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Пристрій: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
else:
    print("Включіть GPU, бо набридне чекати ;)")

Пристрій: cuda
GPU: Tesla T4


## 1. Дані: MRPC

**MRPC** (Microsoft Research Paraphrase Corpus) — пари речень з новинних статей.
Мітка `1` означає, що речення є парафразами, мітка `0` — що ні.

In [4]:
raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
raw_datasets

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

mrpc/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  649kB            

mrpc/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 75.7kB            

mrpc/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  308kB            

mrpc/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [5]:
# Подивимося на один приклад
print("Приклад нееквівалентних речень:")
example = raw_datasets["train"][3]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

print("")
print("Приклад еквівалентних речень:")
example = raw_datasets["train"][5]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

Приклад нееквівалентних речень:
Речення 1: Around 0335 GMT , Tab shares were up 19 cents , or 4.4 % , at A $ 4.56 , having earlier set a record high of A $ 4.57 .
Речення 2: Tab shares jumped 20 cents , or 4.6 % , to set a record closing high at A $ 4.57 .
Мітка:    0 → not_equivalent

Приклад еквівалентних речень:
Речення 1: Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .
Речення 2: With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .
Мітка:    1 → equivalent


### Знайомимось з BERT BASE UNCASED

Для цього прочитайте інформацію на [картці моделі](https://huggingface.co/google-bert/bert-base-uncased)

In [6]:
from transformers import pipeline

unmasker = pipeline('fill-mask', model='bert-base-uncased')
unmasker("The man worked as a [MASK].")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[{'score': 0.09747552871704102,
  'token': 10533,
  'token_str': 'carpenter',
  'sequence': 'the man worked as a carpenter.'},
 {'score': 0.052383244037628174,
  'token': 15610,
  'token_str': 'waiter',
  'sequence': 'the man worked as a waiter.'},
 {'score': 0.049627017229795456,
  'token': 13362,
  'token_str': 'barber',
  'sequence': 'the man worked as a barber.'},
 {'score': 0.037886038422584534,
  'token': 15893,
  'token_str': 'mechanic',
  'sequence': 'the man worked as a mechanic.'},
 {'score': 0.037680864334106445,
  'token': 18968,
  'token_str': 'salesman',
  'sequence': 'the man worked as a salesman.'}]

In [7]:
print("Не по темі, але погляньте, яка \"несправедливість\"...")
unmasker("The woman worked as a [MASK].")

Не по темі, але погляньте, яка "несправедливість"...


[{'score': 0.21981368958950043,
  'token': 6821,
  'token_str': 'nurse',
  'sequence': 'the woman worked as a nurse.'},
 {'score': 0.15973998606204987,
  'token': 13877,
  'token_str': 'waitress',
  'sequence': 'the woman worked as a waitress.'},
 {'score': 0.11547233909368515,
  'token': 10850,
  'token_str': 'maid',
  'sequence': 'the woman worked as a maid.'},
 {'score': 0.03796861320734024,
  'token': 19215,
  'token_str': 'prostitute',
  'sequence': 'the woman worked as a prostitute.'},
 {'score': 0.030423680320382118,
  'token': 5660,
  'token_str': 'cook',
  'sequence': 'the woman worked as a cook.'}]

### Токенізація пари речень

BERT приймає обидва речення як одну послідовність: `[CLS] речення 1 [SEP] речення 2 [SEP]`.
Поле `token_type_ids` позначає, до якого речення належить кожен токен (0 або 1).

Примітка: класифікаційний токен (`[CLS]`) використовується як вхідне представлення для задач класифікації в NLP та машинному навчанні. Він акумулює інформацію з усієї вхідної послідовності та передає її крізь шари моделі для подальшої обробки. Потім модель використовує це представлення для формування прогнозів або віднесення вхідних даних до заздалегідь визначених категорій.

In [8]:
checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
encoded = tokenizer(example["sentence1"], example["sentence2"])

print("")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[:20])
print(encoded["token_type_ids"][:20])
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[-20:])
print(encoded["token_type_ids"][-20:])


['[CLS]', 'revenue', 'in', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.']
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[',', 'revenue', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.', '[SEP]']
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [9]:
def tokenize_function(batch):
    # truncation=True обрізає надто довгі пари до максимальної довжини моделі
    return tokenizer(batch["sentence1"], batch["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Доповнення (padding) робимо динамічно для кожного батча — до найдовшого прикладу в ньому
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_datasets

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

## 2. Модель: кістяк + нова голова

`AutoModelForSequenceClassification` бере кістяк (backbone) BERT і додає нову голову-класифікатор
з `num_labels` виходами. Зверніть увагу на звіт під час завантаження:
- ваги `cls.*` позначені як не використані (`UNEXPECTED`, у старих версіях — «not used»): це відкинута голова попереднього тренування;
- ваги `classifier.*` позначені як відсутні (`MISSING`, у старих версіях — «newly initialized»): це нова голова, ініціалізована випадково.

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

print(model.classifier)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Linear(in_features=768, out_features=2, bias=True)


In [11]:
def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if p.requires_grad or not trainable_only)


print(f"Кістяк (model.bert):   {count_params(model.bert):>12,} параметрів")
print(f"Голова (model.classifier): {count_params(model.classifier):>8,} параметрів")

Кістяк (model.bert):    109,482,240 параметрів
Голова (model.classifier):    1,538 параметрів


## 3. Повне донавчання з `Trainer`

`Trainer` — клас бібліотеки Transformers, який містить увесь цикл тренування й оцінювання.
Метрики рахуємо самі у функції `compute_metrics`.

Оновлюються **всі** ваги: і кістяк, і нова голова.
Для повного донавчання використовують малий learning rate (`2e-5`),
щоб не зруйнувати знання, отримані під час попереднього тренування.

In [12]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
    }


training_args = TrainingArguments(
    output_dir="bert-mrpc",
    num_train_epochs=3,             # зазвичай вистачає 1-3 епохи
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",          # оцінюємо на валідації після кожної епохи
    save_strategy="no",             # чекпоінти не зберігаємо
    logging_steps=50,
    fp16=torch.cuda.is_available(), # змішана точність пришвидшує тренування на GPU
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [13]:
print(f"Параметрів, що тренуються: {count_params(model, trainable_only=True):,}")
trainer.train()
trainer.evaluate()

Параметрів, що тренуються: 109,483,778


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.487028,0.381455,0.838235,0.886986
2,0.328676,0.357605,0.848039,0.893471
3,0.167169,0.416541,0.860294,0.901554


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.167169,0.416541,3,0.860294,0.901554


{'eval_loss': 0.4165409803390503,
 'eval_accuracy': 0.8602941176470589,
 'eval_f1': 0.9015544041450777}

## 4. Перевірка на власних прикладах

In [14]:
def predict(model, sentence1, sentence2):
    model.eval()
    inputs = tokenizer(sentence1, sentence2, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = model(**inputs).logits.softmax(dim=-1)[0]
    return f"парафраз: {probs[1]:.1%}"


pairs = [
    ("The company reported a profit of $5 million.", "The firm earned five million dollars in profit."),
    ("The company reported a profit of $5 million.", "The weather was sunny all week."),
    ("She bought a new car yesterday.", "Yesterday she purchased a new vehicle."),
]
for s1, s2 in pairs:
    print(f"{s1}\n{s2}\n→ {predict(model, s1, s2)}\n")

The company reported a profit of $5 million.
The firm earned five million dollars in profit.
→ парафраз: 86.0%

The company reported a profit of $5 million.
The weather was sunny all week.
→ парафраз: 2.6%

She bought a new car yesterday.
Yesterday she purchased a new vehicle.
→ парафраз: 99.0%



---
# Відповіді на теоретичні питання 0.1 (питання 1–10)

### 1. Як довго виконувалися клітинки ноутбука? Чи були проблеми з запуском?

Ноутбук запускався в Google Colab на GPU **Tesla T4**. Час виконання клітинок:

| Клітинка | Час |
|---|---|
| Імпорти, перевірка GPU | < 1 с |
| Завантаження датасету MRPC | 4 с |
| fill-mask з `bert-base-uncased` (разом із завантаженням моделі 440 MB) | 7 с |
| Токенізація датасету | 1 с |
| Завантаження моделі з новою головою | < 1 с |
| **Донавчання, 3 епохи (690 кроків)** | **≈ 1 хв** |
| Перевірка на власних прикладах | < 1 с |

Увесь ноутбук виконався приблизно за 1.5 хвилини. **Помилок не було**, лише попередження:
- `You are sending unauthenticated requests to the HF Hub` — завантаження з Hugging Face без токена; на роботу не впливає, лише швидкість завантаження може бути нижчою;
- звіт завантаження моделі зі статусами `UNEXPECTED` і `MISSING` — очікувана поведінка, пояснена в розділі 2 ноутбука: відкидається стара голова попереднього тренування (`cls.*`) і створюється нова (`classifier.*`).

### 2. Що таке токен [CLS]? Яким чином він виконує свою функцію?

`[CLS]` (від *classification*) — спеціальний службовий токен, який токенізатор BERT завжди ставить **на початок** послідовності: `[CLS] речення 1 [SEP] речення 2 [SEP]`. Власного змісту він не має, це не слово з тексту.

**Як він виконує свою функцію:**
1. На вході `[CLS]` має свій ембединг, як і будь-який токен.
2. У кожному з 12 шарів трансформера механізм **self-attention** дозволяє кожному токену «дивитися» на всі інші токени послідовності. `[CLS]` теж збирає інформацію з усіх слів обох речень.
3. Оскільки `[CLS]` не прив'язаний до жодного конкретного слова, після 12 шарів його вектор (768 чисел) стає **узагальненим представленням усієї послідовності**.
4. Під час попереднього тренування BERT вектор `[CLS]` використовувався для задачі *Next Sentence Prediction* (чи йде речення 2 одразу після речення 1). Тож модель уже навчилася кодувати в ньому зв'язок між двома реченнями, і це якраз корисно для визначення парафраз.
5. У `AutoModelForSequenceClassification` саме вектор `[CLS]` (після шару pooler: лінійний шар + tanh) подається в нову голову `Linear(768 → 2)`, яка видає оцінки для класів «не парафраз» і «парафраз».

Тобто `[CLS]` — це «точка збору» інформації про весь текст, а голова-класифікатор приймає рішення лише за ним.

### 3. Яке пряме застосування базової моделі bert-base-uncased?

Згідно з карткою моделі, `bert-base-uncased` попередньо тренувалася на двох задачах, і саме їх вона вміє виконувати «з коробки»:
1. **Masked Language Modeling (заповнення пропусків):** передбачити слово на місці `[MASK]`. Це і є пряме застосування, показане в ноутбуці через `pipeline('fill-mask')`: «The man worked as a [MASK].» → carpenter, waiter, barber…
2. **Next Sentence Prediction:** визначити, чи йде одне речення одразу після іншого.

Окрім цього, модель використовують як **кістяк для донавчання** (fine-tuning) на прикладних задачах, де потрібно розуміти весь текст: класифікація текстів (як у цьому демо), розпізнавання іменованих сутностей, відповіді на запитання за текстом. Для **генерації тексту** BERT не призначений: це encoder-модель, яка бачить усе речення одразу. Для генерації використовують decoder-моделі на кшталт GPT.

Також модель *uncased*: перед обробкою весь текст переводиться в нижній регістр, тож вона не розрізняє «Apple» і «apple».

### 4. Чи варто боротися з проявами сексизму LLM-моделей і як?

**Що ми побачили.** На «The man worked as a [MASK].» модель пропонує carpenter (9.7%), waiter, barber, mechanic, salesman. На «The woman worked as a [MASK].» пропонує nurse (22.0%), waitress (16.0%), maid (11.5%), prostitute (3.8%), cook. Тобто для жінок модель обирає вужчий набір професій, переважно обслуговування, і серед них є образливий варіант. Модель відтворює стереотипи з текстів, на яких її тренували (книги й Вікіпедія).

**Чи варто боротися — на мою думку, так.** Модель лише відображає статистику даних, але якщо її використовують для рішень (відбір резюме, рекомендації, автоматичне написання текстів), вона **підсилює** ці стереотипи і може дискримінувати людей. Водночас боротися треба обережно: мета не «стерти» реальний світ, а не дати моделі переносити упередження туди, де стать не має значення.

**Як можна боротися:**
1. **Дані:** балансування навчальних даних, *counterfactual data augmentation* (додавання копій речень з заміненими «він ↔ вона»), фільтрація образливих текстів.
2. **Вимірювання:** спеціальні тести на упередженість (WinoBias, StereoSet, CrowS-Pairs) перед використанням моделі.
3. **Донавчання:** донавчання на збалансованих прикладах; для чат-моделей — навчання з людським зворотним зв'язком (RLHF), де оцінювачі штрафують стереотипні відповіді.
4. **Застосування:** не використовувати модель для рішень щодо людей без перевірки; прибирати ознаки статі з вхідних даних там, де вони не потрібні; людський контроль і прозорість (картка моделі прямо попереджає про ці упередження, і це правильно).

### 5. Для чого в секції «Токенізація пари речень» виводяться числа 0 і 1?

Це поле **`token_type_ids`** (segment IDs): для кожного токена воно позначає, **до якого з двох речень він належить**:
- **0** — токени першого речення разом із `[CLS]` на початку і першим `[SEP]`;
- **1** — токени другого речення разом із фінальним `[SEP]`.

У виводі видно: перші 20 токенів (`[CLS]`, 'revenue', 'in', 'the', 'first', …) мають 0, останні 20 (…, 'earlier', '.', `[SEP]`) мають 1.

**Навіщо:** BERT подає обидва речення як одну послідовність. До ембедингу кожного токена додається ще **ембединг сегмента** (A або B), і так модель розрізняє, де закінчується одне речення й починається інше. Для задачі парафраз це важливо, бо модель має порівнювати саме два окремі речення. Без цього поля їй довелося б орієнтуватися лише за токеном `[SEP]`.

### 6. Що позначають числа 768 і 2 після завантаження моделі?

Вивід `print(model.classifier)`: `Linear(in_features=768, out_features=2, bias=True)` — це нова голова-класифікатор.
- **768** — **розмір прихованого стану** BERT-base (hidden size): кожен токен, зокрема `[CLS]`, після всіх шарів представлений вектором з 768 чисел. Голова приймає на вхід саме вектор `[CLS]`.
- **2** — **кількість класів** (`num_labels=2`): голова видає 2 числа (логіти) для класів `not_equivalent` (0) і `equivalent` (1). Після softmax вони стають імовірностями.

Голова має 768 × 2 ваги + 2 зсуви (bias) = **1 538 параметрів**, що й показано в наступній клітинці.

### 7. Те, що в кістяку набагато більше параметрів, ніж у голові, — добре чи погано?

У кістяку **109 482 240** параметрів, у голові **1 538**, тобто приблизно в 71 000 разів менше.

**У нашому випадку це добре:**
- Усе «розуміння» мови (значення слів, граматика, зв'язки між реченнями) вже закладене в кістяку під час попереднього тренування на мільярдах слів. Голові достатньо навчитися простої речі: як за вектором `[CLS]` вирішити «парафраз / ні».
- Тому вистачає **невеликого датасету** (3 668 пар) і **короткого тренування** (≈ 1 хв): за 3 епохи модель досягла accuracy 86% і F1 0.90. Якби модель тренувалася з нуля, на 3,7 тис. прикладах вона не навчилася б розуміти мову.
- Мала голова легко замінюється під іншу задачу з тим самим кістяком.

**Чого варто остерігатися:** під час повного донавчання оновлюються всі 109 млн параметрів, а прикладів лише кілька тисяч, тож модель може швидко перенавчитися або «забути» знання з попереднього тренування. Тому використовують малий learning rate (2e-5) і лише 1–3 епохи. Перші ознаки перенавчання видно вже на 3-й епосі: validation loss зріс з 0.358 до 0.417, хоча training loss упав з 0.329 до 0.167 (див. питання 9).

### 8. Що саме вимірюється за допомогою accuracy і F1?

Обидві метрики рахуються на **валідаційному наборі MRPC (408 пар речень)** після кожної епохи. Модель для кожної пари передбачає клас (0 — не парафраз, 1 — парафраз), і передбачення порівнюються з правильними мітками.

- **Accuracy** — частка пар, для яких модель угадала клас: 0.860 означає, що правильно визначено 86% пар (≈ 351 з 408).
- **F1** — гармонійне середнє **precision** і **recall** для класу **1 (парафраз)**: `F1 = 2 · P · R / (P + R)`.
  - precision: яка частка пар, які модель назвала парафразами, справді є парафразами;
  - recall: яку частку справжніх парафраз модель знайшла.

**Навіщо дві метрики:** у MRPC класи незбалансовані, близько 68% пар є парафразами. Модель, яка завжди відповідає «парафраз», мала б accuracy ≈ 0.68 і досить високий F1 ≈ 0.81, нічого не вміючи. Тому для MRPC стандартно звітують обидві метрики. Отримані **accuracy 0.860 і F1 0.902** помітно вищі за цей «наївний» рівень.

In [15]:
# Питання 9: тренуємо ту саму модель 8 епох замість 3
import pandas as pd

model_long = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
args_long = TrainingArguments(
    output_dir="bert-mrpc-long",
    num_train_epochs=8,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="no",
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
)
trainer_long = Trainer(
    model=model_long, args=args_long,
    train_dataset=tokenized_datasets["train"], eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator, compute_metrics=compute_metrics,
)
trainer_long.train()

log = pd.DataFrame(trainer_long.state.log_history)
history = log.dropna(subset=["loss"])[["epoch", "loss"]].merge(
    log.dropna(subset=["eval_loss"])[["epoch", "eval_loss", "eval_accuracy", "eval_f1"]], on="epoch")
history.columns = ["Епоха", "Training loss", "Validation loss", "Accuracy", "F1"]
history.round(4)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.572092,0.433237,0.821078,0.880131
2,0.357382,0.419740,0.818627,0.878289
3,0.197734,0.454360,0.872549,0.909091
4,0.111240,0.585663,0.862745,0.905405
5,0.063886,0.780302,0.862745,0.905085
6,0.032024,0.883273,0.860294,0.905158
7,0.018128,0.870659,0.870098,0.910017
8,0.009819,0.898901,0.867647,0.908163


,Епоха,Training loss,Validation loss,Accuracy,F1
0,1.0,0.5721,0.4332,0.8211,0.8801
1,2.0,0.3574,0.4197,0.8186,0.8783
2,3.0,0.1977,0.4544,0.8725,0.9091
3,4.0,0.1112,0.5857,0.8627,0.9054
4,5.0,0.0639,0.7803,0.8627,0.9051
5,6.0,0.0320,0.8833,0.8603,0.9052
6,7.0,0.0181,0.8707,0.8701,0.9100
7,8.0,0.0098,0.8989,0.8676,0.9082


In [16]:
# Питання 10: ще 4 власні пари речень
my_pairs = [
    ("The cat is sleeping on the sofa.", "A cat is napping on the couch."),             # парафраз
    ("The dog bit the man.", "The man bit the dog."),                                   # ті самі слова, інший зміст
    ("The meeting was moved to Monday.", "The meeting was not moved to Monday."),       # заперечення
    ("The price rose by 10 percent.", "The price rose by 20 percent."),                 # різні числа
]
for s1, s2 in my_pairs:
    print(f"{s1}\n{s2}\n→ {predict(model, s1, s2)}\n")

The cat is sleeping on the sofa.
A cat is napping on the couch.
→ парафраз: 98.6%

The dog bit the man.
The man bit the dog.
→ парафраз: 98.1%

The meeting was moved to Monday.
The meeting was not moved to Monday.
→ парафраз: 99.1%

The price rose by 10 percent.
The price rose by 20 percent.
→ парафраз: 98.6%



### 9. Що відбувається при збільшенні кількості епох і чому?

Тренування тієї самої моделі **8 епох** замість 3 (≈ 3 хв 20 с):

| Епоха | Training loss | Validation loss | Accuracy | F1 |
|---|---|---|---|---|
| 1 | 0.5721 | 0.4332 | 0.8211 | 0.8801 |
| 2 | 0.3574 | **0.4197** | 0.8186 | 0.8783 |
| 3 | 0.1977 | 0.4544 | 0.8725 | 0.9091 |
| 4 | 0.1112 | 0.5857 | 0.8627 | 0.9054 |
| 5 | 0.0639 | 0.7803 | 0.8627 | 0.9051 |
| 6 | 0.0320 | 0.8833 | 0.8603 | 0.9052 |
| 7 | 0.0181 | 0.8707 | 0.8701 | 0.9100 |
| 8 | 0.0098 | 0.8989 | 0.8676 | 0.9082 |

**Що відбувається:**
- **Training loss** постійно падає майже до нуля (0.57 → 0.01): модель «завчила» тренувальні приклади.
- **Validation loss** мінімальний на **2-й епосі** (0.42), а далі **зростає більш ніж удвічі** (до 0.90). Це класична ознака **перенавчання** (overfitting).
- **Accuracy і F1** після 3-ї епохи майже не змінюються (0.86–0.87 і 0.905–0.91). Додаткові 5 епох **не покращили** якість, лише витратили час.

**Чому validation loss росте, а accuracy ні.** Loss враховує не лише, чи правильний клас, а й **наскільки модель упевнена**. Перенавчена модель стає надмірно впевненою: на прикладах, де вона помиляється, вона тепер помиляється з імовірністю ~99%, і за такі помилки loss дає дуже великий штраф. Кількість помилок (accuracy) майже та сама, але кожна «коштує» дорожче.

**Причина перенавчання:** 109 млн параметрів і лише 3 668 тренувальних пар. За багато епох модель бачить ті самі приклади знову й знову і починає запам'ятовувати їх замість загальних закономірностей.

**Висновок:** для цієї задачі достатньо **2–3 епох**, як і зазначено в коментарі в ноутбуці («зазвичай вистачає 1–3 епохи»). На практиці перенавчання запобігають через *early stopping*: зупиняють тренування, коли validation loss починає рости, і беруть найкращу епоху (`load_best_model_at_end=True`).

### 10. Чотири перевірки на власних даних

| Речення 1 | Речення 2 | Що перевіряємо | Правильна відповідь | Модель |
|---|---|---|---|---|
| The cat is sleeping on the sofa. | A cat is napping on the couch. | синоніми | парафраз | **98.6%** парафраз ✅ |
| The dog bit the man. | The man bit the dog. | ті самі слова, інший порядок | **не** парафраз | **98.1%** парафраз ❌ |
| The meeting was moved to Monday. | The meeting was not moved to Monday. | заперечення | **не** парафраз | **99.1%** парафраз ❌ |
| The price rose by 10 percent. | The price rose by 20 percent. | різні числа | **не** парафраз | **98.6%** парафраз ❌ |

Модель правильно впоралася лише з синонімами. У трьох «пастках» вона з упевненістю 98–99% назвала парафразами речення **з протилежним або іншим змістом**.

**Чому так:**
1. **Модель спирається на збіг слів.** В усіх трьох хибних прикладах речення майже повністю складаються з тих самих слів. У MRPC пари з великим збігом слів здебільшого справді є парафразами, тож модель засвоїла «зручне правило»: багато спільних слів означає парафраз. Порядок слів, одне слово «not» чи одна цифра на цей збіг майже не впливають.
2. **Незбалансованість даних:** ≈ 68% пар у MRPC є парафразами, тож у сумнівних випадках модель схиляється до класу «парафраз».
3. **Мало прикладів такого типу:** у новинних парах MRPC рідко трапляються штучні «пастки» з перестановкою слів чи запереченням, тому модель не навчилася на них зважати.
4. **Надмірна впевненість** (98–99%) показує, що модель не «сумнівається» в помилкових відповідях, і з високої ймовірності не можна робити висновок, що відповідь правильна.

**Висновок:** accuracy 86% на валідації не означає, що модель «розуміє» зміст речень. Вона добре працює на прикладах, схожих на тренувальні, але легко помиляється на спеціально підібраних випадках. Щоб це виправити, варто додати в тренувальні дані приклади з запереченнями, перестановками й різними числами (adversarial examples).